# Unit Test Generation — Autoresearch
PhD Research: Plain LLM vs Simple RAG vs Iterative Critique RAG for unit test generation.

**Steps:**
1. Mount Google Drive (persist everything across resets)
2. Install dependencies
3. Install & start Ollama, pull model
4. Clone repo
5. One-time setup (download dataset + build knowledge base)
6. Run single experiment
7. Run all 12 experiments (full PhD comparison)
8. Visualize results
9. Cross-task comparison (RQ4)
10. Push results to GitHub

## Step 1: Mount Google Drive
**Run this first.** All results, logs, charts, and checkpoints are saved to both local Colab storage and Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# All PhD outputs go here — edit if you want a different Drive folder
DRIVE_DIR = '/content/drive/MyDrive/PhD_autoresearch'

# Sub-directories (created automatically)
LOGS_DIR        = os.path.join(DRIVE_DIR, 'logs')
PLOTS_DIR       = os.path.join(DRIVE_DIR, 'plots')
RESULTS_DIR     = os.path.join(DRIVE_DIR, 'results')
CHECKPOINTS_DIR = os.path.join(DRIVE_DIR, 'checkpoints')

for d in [LOGS_DIR, PLOTS_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

def save_to_both(src, filename, subdir='results'):
    """Copy a file to both local repo dir and Drive. Prints where it was saved."""
    if not os.path.exists(src):
        print(f'  SKIP {filename} — file not found')
        return
    drive_path = os.path.join(DRIVE_DIR, subdir, filename)
    shutil.copy(src, drive_path)
    print(f'  {filename} → local: {src}')
    print(f'  {filename} → Drive: {drive_path}')

print('Drive mounted. Outputs saved to both locations:')
print(f'  Local:  /content/autoresearch/')
print(f'  Drive:  {DRIVE_DIR}/')

## Step 2: Install Python dependencies

In [ ]:
!pip install -q ollama datasets sentence-transformers scikit-learn nltk rouge beautifulsoup4 numpy pandas matplotlib

## Step 3: Install Ollama and pull model

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama server started.')

In [ ]:
# Choose your model based on available GPU RAM:
# T4  (15GB): llama3.2:latest, phi3:mini, qwen2.5:3b, deepseek-coder:6.7b
# A100(40GB): llama3.1:8b, phi4:14b, qwen2.5:14b

MODEL = 'llama3.2:latest'   # change this if needed
!ollama pull {MODEL}

## Step 4: Clone the repo

In [ ]:
REPO_DIR = '/content/autoresearch'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/balajivenky06/autoresearch.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls

## Step 5: One-time setup (download dataset + build knowledge base)
Only needs to run once per Colab session. The dataset/KB are cached in `~/.cache/autoresearch_unitest/`.

In [ ]:
# Downloads HumanEval + MBPP from HuggingFace (~2 min)
# Fetches pytest/unittest docs and encodes them (~2 min)
!python prepare_unitest.py

## Step 6: Configure and run a single experiment
Use `set_experiment()` to pick METHOD, REASONING, and model. Then run the next cell.

In [ ]:
import re, ast

VALID_METHODS    = {'plain_llm', 'simple_rag', 'iterative_critique'}
VALID_REASONINGS = {'base', 'cot', 'tot', 'got'}

def set_experiment(method='plain_llm', reasoning='base', model=MODEL):
    """Update train_unitest.py config in-place."""
    assert method in VALID_METHODS, f'method must be one of {VALID_METHODS}'
    assert reasoning in VALID_REASONINGS, f'reasoning must be one of {VALID_REASONINGS}'

    with open('train_unitest.py', 'r') as f:
        content = f.read()

    content = re.sub(r'^METHOD\s*=.*$',         'METHOD    = "' + method    + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^REASONING\s*=.*$',       'REASONING = "' + reasoning + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^GENERATOR_MODEL\s*=.*$', 'GENERATOR_MODEL = "' + model + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^HELPER_MODEL\s*=.*$',    'HELPER_MODEL    = "' + model + '"', content, flags=re.MULTILINE)

    try:
        ast.parse(content)
    except SyntaxError as e:
        raise RuntimeError('set_experiment produced invalid Python: ' + str(e))

    with open('train_unitest.py', 'w') as f:
        f.write(content)
    print(f'Config set: method={method}, reasoning={reasoning}, model={model}')

# Set your experiment here:
set_experiment(method='plain_llm', reasoning='base', model=MODEL)

In [ ]:
# Run the experiment — log saved to both local and Drive
log_local = 'run_unitest.log'
log_drive = os.path.join(LOGS_DIR, 'single_run.log')
!python train_unitest.py 2>&1 | tee {log_local}

# Save log + results to Drive
shutil.copy(log_local, log_drive)
save_to_both('results_unitest.tsv', 'results_unitest.tsv')
print('Done.')

In [ ]:
# Quick summary of the last run
!grep -E '^val_score:|^method:|^model:|^avg_' run_unitest.log

## Step 7: Run all 12 experiments (full PhD comparison)
3 methods × 4 reasoning techniques. **~2–4 hours on A100.**

If the runtime resets mid-run, re-run Steps 1–5, then re-run this cell — it will resume from the last checkpoint automatically.

In [ ]:
import subprocess, pandas as pd

EXPERIMENTS = [
    ('plain_llm',          'base'),
    ('plain_llm',          'cot'),
    ('plain_llm',          'tot'),
    ('plain_llm',          'got'),
    ('simple_rag',         'base'),
    ('simple_rag',         'cot'),
    ('simple_rag',         'tot'),
    ('simple_rag',         'got'),
    ('iterative_critique', 'base'),
    ('iterative_critique', 'cot'),
    ('iterative_critique', 'tot'),
    ('iterative_critique', 'got'),
]

results = []
sep = '=' * 55

for method, reasoning in EXPERIMENTS:
    print(f'\n{sep}')
    print(f'  Running: {method} / {reasoning}')
    print(sep)

    set_experiment(method=method, reasoning=reasoning, model=MODEL)

    log_local = f'{method}_{reasoning}.log'
    try:
        ret = subprocess.run(
            ['python', 'train_unitest.py'],
            capture_output=True, text=True, timeout=900
        )
        output = ret.stdout + ret.stderr

        # Save log to both local and Drive
        with open(log_local, 'w') as f:
            f.write(output)
        shutil.copy(log_local, os.path.join(LOGS_DIR, log_local))

    except subprocess.TimeoutExpired:
        print('  TIMEOUT — skipping')
        results.append({'method': f'{method}/{reasoning}', 'val_score': 0.0, 'status': 'timeout'})
        continue

    # Parse key metrics from stdout
    row = {'method': f'{method}/{reasoning}', 'model': MODEL, 'status': 'ok'}
    for line in output.splitlines():
        for key in ['val_score', 'avg_noise_rate', 'avg_faithfulness',
                    'avg_retrieval_secs', 'avg_llm_secs', 'avg_syntax_valid',
                    'avg_edge_coverage', 'avg_rouge_1']:
            if line.strip().startswith(key + ':'):
                try:
                    row[key] = float(line.split(':')[1].strip())
                except Exception:
                    pass
    if row.get('val_score', 0.0) == 0.0:
        row['status'] = 'crash'
    results.append(row)
    print(f'  val_score: {row.get("val_score", 0):.6f}  [{row["status"]}]')

    # Save results_unitest.tsv to both local and Drive after every experiment
    save_to_both('results_unitest.tsv', 'results_unitest.tsv')

# Summary table
df = pd.DataFrame(results).sort_values('val_score', ascending=False)
print(f'\n{sep}')
print('  FINAL RESULTS')
print(sep)
print(df.to_string(index=False))

# Save summary CSV to both local and Drive
df.to_csv('summary_all_experiments.csv', index=False)
save_to_both('summary_all_experiments.csv', 'summary_all_experiments.csv')
print('Done.')

## Step 8: Visualize results
Generates 7 KPI charts — saved to both `plots_unitest/` (local) and Drive.

In [ ]:
from IPython.display import Image, display

# Generate all charts into plots_unitest/
!python visualize_unitest.py

# Save charts to Drive and display
charts = [
    'heatmap.png',
    'grouped_bar.png',
    'radar.png',
    'per_metric_bar.png',
    'noise_rate.png',
    'cost_breakdown.png',
    'faithfulness.png',
]

for fname in charts:
    src = os.path.join('plots_unitest', fname)
    save_to_both(src, fname, subdir='plots')
    if os.path.exists(src):
        display(Image(src))

## Step 9: Cross-task comparison — RQ4
Requires `results_docstring.tsv` from the RAG-Docstring repo.
Upload it to `PhD_autoresearch/results/` in Drive, then run this cell.

In [ ]:
# Load results_docstring.tsv from Drive into local working dir
docstring_tsv_drive = os.path.join(RESULTS_DIR, 'results_docstring.tsv')
if os.path.exists(docstring_tsv_drive):
    shutil.copy(docstring_tsv_drive, 'results_docstring.tsv')
    print('results_docstring.tsv loaded from Drive.')
else:
    print(f'Not found: {docstring_tsv_drive}')
    print('Upload results_docstring.tsv to Drive first, then re-run this cell.')

In [ ]:
from IPython.display import Image, display

# Generate cross-task comparison charts
!python compare_tasks.py

# Save to both local plots_compare/ and Drive
compare_charts = ['faithfulness_by_task.png', 'noise_vs_faithfulness.png', 'pareto.png']
for fname in compare_charts:
    src = os.path.join('plots_compare', fname)
    save_to_both(src, f'compare_{fname}', subdir='plots')
    if os.path.exists(src):
        display(Image(src))

# Save cross-task summary table
save_to_both('plots_compare/summary_table.tsv', 'cross_task_summary.tsv')

## Step 10: Push results to GitHub
Commits all results, logs, and charts from local Colab storage to GitHub.

**Before running:** set your GitHub token below.
Get one at: GitHub → Settings → Developer Settings → Personal Access Tokens → Tokens (classic) → New token (scope: `repo`)

In [ ]:
# ---- Set your GitHub credentials ----
GITHUB_TOKEN = 'ghp_xxxxxxxxxxxxxxxxxxxx'   # paste your token here
GITHUB_EMAIL = 'your@email.com'
GITHUB_NAME  = 'Your Name'
# -------------------------------------

import subprocess

!git config user.email "{GITHUB_EMAIL}"
!git config user.name  "{GITHUB_NAME}"

# Stage all results, logs, and charts
files_to_commit = [
    'results_unitest.tsv',
    'summary_all_experiments.csv',
    'plots_unitest/',
    'plots_compare/',
    '*.log',
]

# Remove results_unitest.tsv from .gitignore so it can be committed
with open('.gitignore', 'r') as f:
    gi = f.read()
if 'results_unitest.tsv' in gi:
    gi = gi.replace('results_unitest.tsv', '# results_unitest.tsv  (committed for PhD)')
    with open('.gitignore', 'w') as f:
        f.write(gi)
    print('Unblocked results_unitest.tsv from .gitignore')

for f in files_to_commit:
    !git add -f {f} 2>/dev/null || true

# Commit
commit_msg = f'add experiment results: {MODEL} — all 12 methods'
result = subprocess.run(['git', 'commit', '-m', commit_msg], capture_output=True, text=True)
print(result.stdout or result.stderr)

# Push using token
remote_url = f'https://{GITHUB_TOKEN}@github.com/balajivenky06/autoresearch.git'
push = subprocess.run(['git', 'push', remote_url, 'master'], capture_output=True, text=True)
if push.returncode == 0:
    print('Pushed to GitHub successfully.')
else:
    print('Push failed:', push.stderr)